# Earth Engine exports for the Rural Hawaiʻi Climate Resilience Observatory

Colab version of the scripts in `scripts/gee/`. Running every cell produces three
CSVs and downloads them to your computer:

| File | Save it as | What it is |
|---|---|---|
| `pukalani_rainfall_monthly.csv` | `data/research/pukalani_rainfall_monthly.csv` | Monthly rain at the study area, 2018–2024, from CHIRPS, TerraClimate, IMERG, ERA5-Land |
| `lcms_site_classes.csv` | `data/research/lcms_site_classes.csv` | LCMS change / land cover / land use at each GLOBE site, 2018–2024 |
| `landcover_site_classes.csv` | `data/research/landcover_site_classes.csv` | ESA WorldCover (2020, 2021) and Dynamic World (2018–2024) at each GLOBE site |

**You need** a Google account registered for Earth Engine and a Cloud project ID
for it (code.earthengine.google.com → your project name at the top right).

Written without Earth Engine access, so not yet run. If a cell errors, the message
names the dataset or band; the catalog page for that ID shows the current name.
NLCD is not used: it does not cover Hawaiʻi.

## 1. Sign in

In [ ]:
import ee
import pandas as pd

PROJECT = "your-earth-engine-project-id"  # @param {type:"string"}

ee.Authenticate()            # opens a sign-in link the first time
ee.Initialize(project=PROJECT)
print("Earth Engine ready:", ee.String("ok").getInfo())

In [ ]:
def to_df(fc, columns):
    """A small FeatureCollection as a DataFrame (these tables are tens to hundreds of rows)."""
    rows = [f["properties"] for f in fc.getInfo()["features"]]
    return pd.DataFrame(rows, columns=columns)


def save(df, name):
    df.to_csv(name, index=False)
    print(f"saved {name}: {len(df)} rows")
    try:
        from google.colab import files
        files.download(name)
    except ImportError:
        pass  # not in Colab: the file is in the working directory

## 2. Rainfall at the study area

Each value is the one grid pixel (~4.6–11 km) containing the study-area centre:
the area's rainfall regime, not a gauge at Pukalani. `-9999` = no data.
CHIRPS keeps a day count so a part month is visible.

In [ ]:
CENTRE = ee.Geometry.Point([-156.344856, 20.838722])   # same point as every script
START = ee.Date("2018-01-01")
N_MONTHS = 84                                          # 2018-01 .. 2024-12
MISSING = -9999

chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")            # mm/day
terra = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE").select("pr")                    # mm/month
imerg = ee.ImageCollection("NASA/GPM_L3/IMERG_MONTHLY_V07").select("precipitation")     # mm/hour (monthly mean)
era5 = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR").select("total_precipitation_sum")  # metres

for name, col in [("CHIRPS", chirps), ("TerraClimate", terra), ("IMERG", imerg), ("ERA5-Land", era5)]:
    c = col.filterDate(START, START.advance(N_MONTHS, "month"))
    print(name, c.size().getInfo(), "images",
          ee.Date(c.aggregate_min("system:time_start")).format("YYYY-MM").getInfo(), "to",
          ee.Date(c.aggregate_max("system:time_start")).format("YYYY-MM").getInfo())


def at(img, band, scale):
    # An empty month reduces to an empty dictionary, so the default comes back.
    return ee.Dictionary(img.reduceRegion(reducer=ee.Reducer.first(), geometry=CENTRE, scale=scale)).get(band, MISSING)


def month_row(m):
    s = START.advance(m, "month")
    e = s.advance(1, "month")
    days = chirps.filterDate(s, e)
    return ee.Feature(None, {
        "date": s.format("YYYY-MM-dd"),
        "chirps_mm": at(days.sum(), "precipitation", 5566),
        "chirps_days": days.size(),
        "terraclimate_mm": at(terra.filterDate(s, e).mean(), "pr", 4638),
        "imerg_mm": at(imerg.filterDate(s, e).mean().multiply(e.difference(s, "hour")), "precipitation", 11132),
        "era5land_mm": at(era5.filterDate(s, e).mean().multiply(1000), "total_precipitation_sum", 11132),
    })


RAIN_COLUMNS = ["date", "chirps_mm", "chirps_days", "terraclimate_mm", "imerg_mm", "era5land_mm"]
rain = to_df(ee.FeatureCollection(ee.List.sequence(0, N_MONTHS - 1).map(month_row)), RAIN_COLUMNS)
rain.head(12)

In [ ]:
# Quick look before saving: the four products should rise and fall together.
rain.replace(MISSING, float("nan")).plot(
    x="date", y=["chirps_mm", "terraclimate_mm", "imerg_mm", "era5land_mm"],
    figsize=(12, 4), title="Monthly rainfall at the Pukalani study area (mm)")
save(rain, "pukalani_rainfall_monthly.csv")

## 3. GLOBE sites

Same 15 Pukalani-area sites as `scripts/gee/lcms_site_sample.js` (from `scripts/export_sites_for_gee.py --area pukalani --complete-only`). Paste a new list here to change them.

In [ ]:
SITES = [  # (lon, lat, GLOBE site name)
    (-156.44900, 20.74890, '04QGH656964'),
    (-156.43800, 20.70490, '04QGH668915'),
    (-156.33350, 20.72540, '04QGH777940'),
    (-156.33280, 20.74120, '04QGH777957'),
    (-156.33240, 20.74270, '04QGH777959'),
    (-156.32970, 20.75880, '04QGH780977'),
    (-156.34520, 20.83810, '04QGJ762065'),
    (-156.34459, 20.83841, '04QGJ763065'),
    (-156.34395, 20.83800, '04QGJ764064'),
    (-156.34400, 20.83860, '04QGJ764065'),
    (-156.34360, 20.84050, '04QGJ764067'),
    (-156.31680, 20.85070, '04QGJ792079'),
    (-156.31660, 20.85120, '04QGJ792080'),
    (-156.29400, 20.84160, '04QGJ816069'),
    (-156.29300, 20.84280, '04QGJ817071'),
]
sites = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lon, lat]), {"site": name}) for lon, lat, name in SITES])
print(sites.size().getInfo(), "sites")

## 4. LCMS at each site (USDA Forest Service, 30 m)

Exports class codes **and** names. Tries the newer LCMS asset first; prints what it found so a mismatch is visible.


In [ ]:
# Google has superseded USFS/GTAC/LCMS/v2024-10, and filtering it by
# study_area == "HI" matched nothing (LCMS years: []). So: try the new asset
# first, filter by LOCATION (works whatever the region property is called),
# and stop with a clear message if neither covers the sites.
CANDIDATES = [
    "projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11",  # named in Google's deprecation notice
    "USFS/GTAC/LCMS/v2024-10",
]
lcms = None
for asset in CANDIDATES:
    try:
        col = ee.ImageCollection(asset).filterBounds(sites)
        n = col.size().getInfo()
    except ee.EEException as exc:
        print(asset, "->", exc)
        continue
    print(asset, "->", n, "images over the GLOBE sites")
    if n:
        lcms = col
        break
if lcms is None:
    raise RuntimeError(
        "No LCMS images cover the GLOBE sites in either asset. Search the Earth Engine "
        "catalog for 'LCMS' + 'Hawaii' and put that asset ID first in CANDIDATES.")

first = ee.Image(lcms.first())
BANDS = ["Change", "Land_Cover", "Land_Use"]
print("bands:", first.bandNames().getInfo())
props = first.propertyNames().getInfo()
named = [b for b in BANDS if f"{b}_class_values" in props and f"{b}_class_names" in props]
print("bands with class-name tables:", named)
years = sorted(set(ee.List(lcms.aggregate_array("system:time_start"))
                   .map(lambda t: ee.Date(t).get("year")).getInfo()))
years = [y for y in years if 2018 <= y <= 2024]
print("LCMS years:", years)


def with_name(band):
    lookup = ee.Dictionary.fromLists(
        ee.List(first.get(band + "_class_values")).map(lambda v: ee.Number(v).format("%d")),
        ee.List(first.get(band + "_class_names")))
    return lambda f: f.set(band + "_name", lookup.get(ee.Number(f.get(band)).format("%d"), "unknown"))


def lcms_year(y):
    # mosaic(): a year can be split over several images (tiles / regions).
    img = lcms.filterDate(f"{y}-01-01", f"{y + 1}-01-01").mosaic().select(BANDS)
    return img.sampleRegions(collection=sites.map(lambda f: f.set("year", y)), scale=30, geometries=False)


lcms_rows = lcms_year(years[0])
for y in years[1:]:
    lcms_rows = lcms_rows.merge(lcms_year(y))
for band in named:
    lcms_rows = lcms_rows.map(with_name(band))
if "Land_Cover" not in named:
    print("WARNING: no Land_Cover class names in this asset; the CSV will hold codes only and "
          "scripts/crosscheck_lcms.py will refuse it. Send the printed properties to fix the lookup.")

LCMS_COLUMNS = ["site", "year"] + BANDS + [f"{b}_name" for b in named]
lcms_df = to_df(lcms_rows, LCMS_COLUMNS)
save(lcms_df, "lcms_site_classes.csv")
lcms_df.head(10)


## 5. ESA WorldCover and Dynamic World at each site (10 m)

WorldCover exists for 2020 (v100) and 2021 (v200). Dynamic World is reduced to the
most common class at each site in each year, so it is the year's typical cover.

In [ ]:
WC_NAMES = {10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland", 50: "Built-up",
            60: "Bare / sparse vegetation", 70: "Snow and ice", 80: "Permanent water bodies",
            90: "Herbaceous wetland", 95: "Mangroves", 100: "Moss and lichen"}
DW_NAMES = ["water", "trees", "grass", "flooded_vegetation", "crops",
            "shrub_and_scrub", "built", "bare", "snow_and_ice"]


def sample(img, band, product, year):
    fc = img.sampleRegions(collection=sites, scale=10, geometries=False).filter(ee.Filter.notNull([band]))
    return [{"site": p["site"], "product": product, "year": year, "class_value": int(p[band])}
            for p in (f["properties"] for f in fc.getInfo()["features"])]


rows = []
for asset, year in (("ESA/WorldCover/v100", 2020), ("ESA/WorldCover/v200", 2021)):
    rows += sample(ee.ImageCollection(asset).first().select("Map"), "Map", "worldcover", year)
for year in range(2018, 2025):
    s = ee.Date.fromYMD(year, 1, 1)
    modal = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(sites)
             .filterDate(s, s.advance(1, "year")).select("label").mode())
    rows += sample(modal, "label", "dynamicworld", year)

landcover = pd.DataFrame(rows, columns=["site", "product", "year", "class_value"])
landcover["class_name"] = [WC_NAMES.get(v, "unknown") if p == "worldcover" else DW_NAMES[v]
                           for p, v in zip(landcover["product"], landcover["class_value"])]
save(landcover, "landcover_site_classes.csv")
landcover.pivot_table(index="site", columns=["product", "year"], values="class_name", aggfunc="first")

## 6. Next, on your computer

```bash
python -m lib.rainfall
python scripts/crosscheck_lcms.py --csv data/research/lcms_site_classes.csv \
    --landcover data/research/landcover_site_classes.csv
```